In [63]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder


In [64]:
df2 = pd.read_csv("dataset2.csv")

In [58]:
missing_values = df2.isnull().sum()
print("Missing Values per Column:")
print(missing_values)

Missing Values per Column:
Date               0
Time               0
Latitude           0
Longitude          0
Depth              0
Magnitude          0
Magt               0
No stations        0
Gap                0
Dist to Closest    0
RMS                0
SRC                0
EventID            0
dtype: int64


In [59]:
print(df2.columns)

Index(['Date', 'Time', 'Latitude', 'Longitude', 'Depth', 'Magnitude', 'Magt',
       'No stations', 'Gap', 'Dist to Closest', 'RMS', 'SRC', 'EventID'],
      dtype='object')


In [60]:
df_scaled = df2.copy() 

for column in df_scaled.select_dtypes(include=np.number).columns: 
    df_scaled[column] = (df_scaled[column] - df_scaled[column].min()) / (df_scaled[column].max() - df_scaled[column].min())     
  

print(df_scaled)

             Date         Time  Latitude  Longitude     Depth  Magnitude Magt  \
0      1966/07/01  09:41:21.82  0.305023   0.456954  0.101063   0.045558   Mx   
1      1966/07/02  12:08:34.25  0.293488   0.466272  0.074108   0.159453   Mx   
2      1966/07/02  12:16:14.95  0.293929   0.465701  0.081444   0.091116   Mx   
3      1966/07/02  12:25:06.12  0.294233   0.466162  0.074932   0.022779   Mx   
4      1966/07/05  18:54:54.36  0.303288   0.457701  0.064793   0.022779   Mx   
...           ...          ...       ...        ...       ...        ...  ...   
18025  2007/12/19  12:14:09.62  0.174756   0.683437  0.057951   0.241458   ML   
18026  2007/12/21  12:14:56.45  0.403418   0.378804  0.069821   0.018223   ML   
18027  2007/12/23  21:43:43.54  0.396545   0.628839  0.082433   0.123007   ML   
18028  2007/12/28  01:59:42.40  0.347149   0.415181  0.049378   0.009112   ML   
18029  2007/12/28  23:20:28.12  0.509164   0.309744  0.019289   0.091116   Mw   

       No stations       Ga

In [61]:
df_numeric = df2.select_dtypes(include=[np.number])
corr = df_numeric.corr()
corr.style.background_gradient(cmap='coolwarm').format("{:.2f}")

,Latitude,Longitude,Depth,Magnitude,No stations,Gap,Dist to Closest,RMS,EventID
Latitude,1.00,-0.69,0.18,-0.05,-0.02,0.24,0.22,0.03,0.05
Longitude,-0.69,1.00,-0.19,0.04,-0.12,-0.11,-0.15,-0.00,0.00
Depth,0.18,-0.19,1.00,0.06,-0.10,0.20,0.04,-0.01,-0.00
Magnitude,-0.05,0.04,0.06,1.00,0.17,0.07,0.11,0.03,-0.01
No stations,-0.02,-0.12,-0.10,0.17,1.00,-0.36,-0.05,0.01,0.21
Gap,0.24,-0.11,0.20,0.07,-0.36,1.00,0.66,0.06,0.00
Dist to Closest,0.22,-0.15,0.04,0.11,-0.05,0.66,1.00,0.06,0.12
RMS,0.03,-0.00,-0.01,0.03,0.01,0.06,0.06,1.00,0.00
EventID,0.05,0.00,-0.00,-0.01,0.21,0.00,0.12,0.00,1.00


In [68]:
non_numeric_columns = df2.select_dtypes(exclude=[np.number]).columns
print("Non-Numeric Columns:", non_numeric_columns)

label_encoder = LabelEncoder()
for col in non_numeric_columns:
    if col != 'Date':
        df2[col] = label_encoder.fit_transform(df2[col])

df2['Date'] = pd.to_datetime(df2['Date'], format='%Y/%m/%d') 
print(df2['Date'].dtype)

df2['timestamp'] = df2['Date'].astype('int64') / 10**9  # Convert to seconds since UNIX epoch
print(df2['timestamp'])

Non-Numeric Columns: Index(['Date'], dtype='object')
datetime64[ns]
0       -1.105920e+08
1       -1.105056e+08
2       -1.105056e+08
3       -1.105056e+08
4       -1.102464e+08
             ...     
18025    1.198022e+09
18026    1.198195e+09
18027    1.198368e+09
18028    1.198800e+09
18029    1.198800e+09
Name: timestamp, Length: 18030, dtype: float64


In [69]:
corr = df2.corr()
corr.style.background_gradient(cmap='coolwarm').format("{:.2f}")

c:\Users\nadez\anaconda3\Lib\site-packages\pandas\io\formats\style.py:3807: RuntimeWarning: All-NaN slice encountered
  smin = np.nanmin(gmap) if vmin is None else vmin
c:\Users\nadez\anaconda3\Lib\site-packages\pandas\io\formats\style.py:3808: RuntimeWarning: All-NaN slice encountered
  smax = np.nanmax(gmap) if vmax is None else vmax


,Date,Time,Latitude,Longitude,Depth,Magnitude,Magt,No stations,Gap,Dist to Closest,RMS,SRC,EventID,timestamp
Date,1.00,-0.00,0.06,0.07,0.04,-0.02,-0.05,0.28,0.10,0.22,0.01,nan,0.72,1.00
Time,-0.00,1.00,-0.00,0.00,-0.02,0.01,-0.00,-0.01,-0.02,-0.02,-0.00,nan,0.01,-0.00
Latitude,0.06,-0.00,1.00,-0.69,0.18,-0.05,-0.13,-0.02,0.24,0.22,0.03,nan,0.05,0.06
Longitude,0.07,0.00,-0.69,1.00,-0.19,0.04,0.12,-0.12,-0.11,-0.15,-0.00,nan,0.00,0.07
Depth,0.04,-0.02,0.18,-0.19,1.00,0.06,-0.03,-0.10,0.20,0.04,-0.01,nan,-0.00,0.04
Magnitude,-0.02,0.01,-0.05,0.04,0.06,1.00,-0.14,0.17,0.07,0.11,0.03,nan,-0.01,-0.02
Magt,-0.05,-0.00,-0.13,0.12,-0.03,-0.14,1.00,-0.22,0.12,0.11,0.01,nan,0.02,-0.05
No stations,0.28,-0.01,-0.02,-0.12,-0.10,0.17,-0.22,1.00,-0.36,-0.05,0.01,nan,0.21,0.28
Gap,0.10,-0.02,0.24,-0.11,0.20,0.07,0.12,-0.36,1.00,0.66,0.06,nan,0.00,0.10
Dist to Closest,0.22,-0.02,0.22,-0.15,0.04,0.11,0.11,-0.05,0.66,1.00,0.06,nan,0.12,0.22


In [70]:
df2['year'] = df2['Date'].dt.year
df2['month'] = df2['Date'].dt.month
df2['day'] = df2['Date'].dt.day
df2['hour'] = df2['Date'].dt.hour
df2['minute'] = df2['Date'].dt.minute
df2['day_of_week'] = df2['Date'].dt.dayofweek
df2['quarter'] = df2['Date'].dt.quarter

In [79]:
# features = ['year', 'month', 'day', 'hour', 'minute', 'day_of_week', 'quarter', 'Longitude', 'Latitude']
features = ['Longitude', 'Latitude', 'Depth', 'Magt', 'RMS', 'timestamp']
target = 'Magnitude'

X = df2[features]
y = df2[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.dtypes)  


Longitude    float64
Latitude     float64
Depth        float64
Magt           int32
RMS          float64
timestamp    float64
dtype: object


In [80]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(comparison_df)

       Actual  Predicted
15928    3.01     3.1132
7753     3.22     3.3140
7944     3.67     3.7878
1020     3.29     3.5494
2864     3.33     3.3645
...       ...        ...
12444    3.05     4.1672
14896    3.12     3.0742
7112     3.50     3.5972
2913     3.01     3.2077
3351     3.59     3.2755

[5409 rows x 2 columns]


In [78]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R²): {r2:.2f}")

UFuncTypeError: ufunc 'subtract' cannot use operands with types dtype('float64') and dtype('<M8[ns]')

In [75]:
rf_model = RandomForestRegressor(n_estimators=100)
scores = cross_val_score(rf_model, X, y, cv=5)
print(f"Cross-Validation Scores: {scores}")
print(f"Average Score: {scores.mean()}")


Cross-Validation Scores: [ -0.24668661   0.1802763    0.20499112   0.02379703 -11.08423357]
Average Score: -2.1843711464228206
